In [2]:

import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.utils import save_image


# ==========================================
# Step 1: 配置
# ==========================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

IMAGE_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-4
STYLE_WEIGHT = 10.0

CONTENT_DIR = Path("data/coco")
STYLE_DIR = Path("data/wikiart")

torch.manual_seed(42)


# ==========================================
# Step 2: Dataset 图片读取器
# ==========================================

class StyleDataset(Dataset):

    def __init__(self, content_paths, style_paths, transform):
        self.content_paths = list(content_paths)
        self.style_paths = list(style_paths)
        self.transform = transform

        if not self.content_paths or not self.style_paths:
            raise ValueError("Both domains need images")

    def __len__(self):
        return max(
            len(self.content_paths),
            len(self.style_paths)
        )

    def __getitem__(self, index):

        # 按 index 读取内容图
        content_path = self.content_paths[
            index % len(self.content_paths)
        ]

        # 随机读取一张风格图
        style_path = random.choice(self.style_paths)

        with Image.open(content_path) as img:
            content = self.transform(img.convert("RGB"))

        with Image.open(style_path) as img:
            style = self.transform(img.convert("RGB"))

        return {
            "content": content,
            "style": style
        }


transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])


def get_image_paths(folder):
    extensions = {".jpg", ".jpeg", ".png", ".webp"}
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in extensions
    )


content_paths = get_image_paths(CONTENT_DIR)
style_paths = get_image_paths(STYLE_DIR)

dataset = StyleDataset(
    content_paths,
    style_paths,
    transform
)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)


# ==========================================
# Step 3: Encoder 提取特征
# ==========================================

class Encoder(nn.Module):

    def __init__(self):
        super().__init__()

        weights = models.VGG19_Weights.DEFAULT
        vgg = models.vgg19(weights=weights).features

        # VGG19 不同深度的特征
        self.layers = nn.ModuleList([
            vgg[:2],
            vgg[2:7],
            vgg[7:12],
            vgg[12:21]
        ])

        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406])
            .view(1, 3, 1, 1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225])
            .view(1, 3, 1, 1)
        )

        # 冻结预训练 Encoder
        for param in self.parameters():
            param.requires_grad = False

    def forward(self, x):

        # VGG19 需要 ImageNet 标准化
        x = (x - self.mean) / self.std

        features = []

        for layer in self.layers:
            x = layer(x)
            features.append(x)

        return features


# ==========================================
# Step 4: AdaIN 特征融合
# ==========================================

def calc_mean_std(x, eps=1e-5):

    mean = x.mean(dim=(2, 3), keepdim=True)

    var = x.var(
        dim=(2, 3),
        keepdim=True,
        unbiased=False
    )

    std = torch.sqrt(var + eps)

    return mean, std


def adain(content_features, style_features):

    c_mean, c_std = calc_mean_std(content_features)
    s_mean, s_std = calc_mean_std(style_features)

    # 先标准化内容，再应用风格统计量
    normalized = (
        content_features - c_mean
    ) / c_std

    return normalized * s_std + s_mean


# ==========================================
# Step 5: Decoder 生成图片
# ==========================================

class Decoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            nn.Conv2d(512, 256, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 3, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


# ==========================================
# Step 6: AdaIN 风格迁移模型
# ==========================================

class StyleTransferModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, content, style, alpha=1.0):

        # 提取特征
        content_features = self.encoder(content)
        style_features = self.encoder(style)

        # 最深层的特征
        c = content_features[-1]
        s = style_features[-1]

        # AdaIN 融合
        target = adain(c, s)

        # 控制风格强度
        target = alpha * target + (1-alpha) * c

        # 生成图片
        output = self.decoder(target)

        return output, target, style_features


# ==========================================
# Step 7: Loss 损失函数
# ==========================================

def content_loss(output_features, target):
    return F.mse_loss(output_features[-1], target)


def style_loss(output_features, style_features):

    loss = 0.0

    for output, style in zip(
        output_features,
        style_features
    ):
        output_mean, output_std = calc_mean_std(output)
        style_mean, style_std = calc_mean_std(style)

        loss += F.mse_loss(output_mean, style_mean)
        loss += F.mse_loss(output_std, style_std)

    return loss


# ==========================================
# Step 8: 创建模型和 Optimizer
# ==========================================

model = StyleTransferModel().to(DEVICE)

model.encoder.eval()
model.decoder.train()

optimizer = torch.optim.Adam(
    model.decoder.parameters(),
    lr=LR
)


# ==========================================
# Step 9: Training Loop
# ==========================================

Path("outputs").mkdir(exist_ok=True)

for epoch in range(EPOCHS):

    total_loss = 0.0

    for batch in train_loader:

        content = batch["content"].to(DEVICE)
        style = batch["style"].to(DEVICE)

        optimizer.zero_grad()

        # 前向传播
        output, target, style_features = model(
            content,
            style
        )

        # 对输出图片再次提取特征
        output_features = model.encoder(output)

        # 计算损失
        loss_c = content_loss(
            output_features,
            target.detach()
        )

        loss_s = style_loss(
            output_features,
            [f.detach() for f in style_features]
        )

        loss = loss_c + STYLE_WEIGHT * loss_s

        # 反向传播
        loss.backward()

        # 更新 Decoder 参数
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS}, "
        f"Loss: {avg_loss:.4f}"
    )

    # 每个 epoch 保存一张结果图片
    save_image(
        output[0].detach().cpu(),
        f"outputs/epoch_{epoch+1}.png"
    )

# 保存模型
torch.save(
    model.decoder.state_dict(),
    "decoder.pth"
)

print("Training finished!")


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /Users/haruka/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:19<00:00, 29.3MB/s] 


Epoch 1/10, Loss: 80.8159
Epoch 2/10, Loss: 51.0245
Epoch 3/10, Loss: 45.2144
Epoch 4/10, Loss: 42.4236


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/epoch_4.png'

In [ ]:

# 需先运行上面的模型类定义

model = StyleTransferModel().to(DEVICE)

# 加载训练好的 Decoder
model.decoder.load_state_dict(
    torch.load("decoder.pth", map_location=DEVICE,
               weights_only=True)
)

model.eval()

def load_image(path):
    with Image.open(path) as img:
        image = transform(img.convert("RGB"))
    return image.unsqueeze(0).to(DEVICE)


content = load_image("my_photo.jpg")
style = load_image("my_style.jpg")

with torch.no_grad():
    output, _, _ = model(
        content,
        style,
        alpha=1.0
    )

save_image(output[0].cpu(), "result.png")
